### Exhaustive 3: Tools (Function Calling) Deep Dive

This notebook breaks down `3-tools.py`, where the model gets live weather data by asking **your code** to run a function.

**The key point:** the model never runs code; it can only write text. "Calling a tool" means the model writes a small request like "please run `get_weather` with these two numbers", and your Python code does the actual running. That's why one tool use needs **two API calls with your code in between**.

##### The whole flow:
```
1. Call 1: client.chat.completions.create(messages, tools)
   The model reads the question + the tool's name/description/parameters
   and replies with a REQUEST, not an answer:
   name='get_weather', arguments='{"latitude":59.9139,"longitude":10.7522}'
            │
            ▼
2. Your code (no model involved):
   json.loads(arguments) -> call_function -> get_weather(**args)
   -> real HTTP request to api.open-meteo.com -> {"temperature_2m": 14.5, ...}
   Append the model's request and the result to `messages`.
            │
            ▼
3. Call 2: client.chat.completions.parse(messages, tools, response_format)
   The model reads the whole history, including the result,
   and writes the answer -> WeatherResponse(temperature=14.5, response='...')
```

- **Section 1**: What the model reads: the tool `description`, not the docstring.
- **Section 2**: Reading the response of call 1.
- **Section 3**: Call 1 did not fetch the weather, so why the rest?
- **Section 4**: `call_function`, `**args`, and the loop line by line.
- **Section 5**: The four messages the second call receives.
- **Section 6**: Call 2, and where `Field()` descriptions go.
- **Section 7**: Checking the final answer against what the model was given.

**Note (2026-09-20):** the code in this notebook has two fixes that aren't in the original course code:
- the assistant message is appended once, before the loop (Section 4);
- `get_weather` asks for local time and m/s, and returns the units and timezone with the values (Section 7).

The outputs saved in this notebook come from the run **before** those fixes, and the explanations below refer to them. `Supplement_3-tools.ipynb` has a fresh run with the fixes.

In [2]:
import json
import os
from pprint import pprint
import requests
from openai import OpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv

In [4]:
load_dotenv()
client = OpenAI()

In [5]:
def get_weather(latitude, longitude):
    """This is a publically available API that returns the weather for a given location."""
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m&timezone=auto&wind_speed_unit=ms"
    )
    data = response.json()
    # send the units and timezone too, so the model doesn't have to guess them
    return {
        "current": data["current"],
        "units": data["current_units"],
        "timezone": data["timezone"],
    }

In [6]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current temperature for provided coordinates in celsius.",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {"type": "number"},
                    "longitude": {"type": "number"},
                },
                "required": ["latitude", "longitude"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]


#### 1. What the model reads: the tool `description`, not the docstring

**Key point:** the model reads only the JSON request body that the SDK sends, and the Python function `get_weather` is never part of it. Its docstring never leaves your machine. Everything the model knows about the tool comes from the `tools` list, so the text that plays the docstring's role is `"description"`.

Look at the call in the next cell: `create(model=..., messages=..., tools=...)`. `get_weather` isn't passed anywhere. The only link between the model and your function is the **name string** `"get_weather"`, which you wrote in both places.

##### The exact request body of call 1
This is what the SDK sends for the next cell. I captured it with a fake HTTP transport, so nothing went to OpenAI. The docstring text ("This is a publically available API...") appears nowhere in it.

<div style="font-size: 0.85em">

```json
{
  "messages": [
    {"role": "system", "content": "You are a helpful weather assistant."},
    {"role": "user", "content": "What's the weather like in Oslo today?"}
  ],
  "model": "gpt-5-nano",
  "tools": [{
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get current temperature for provided coordinates in celsius.",
      "parameters": {
        "type": "object",
        "properties": {"latitude": {"type": "number"}, "longitude": {"type": "number"}},
        "required": ["latitude", "longitude"],
        "additionalProperties": false
      },
      "strict": true
    }
  }]
}
```

</div>

##### What the model uses each part of the tool for
- **`name`**: the label the model writes back when it wants this tool. Your `call_function` matches on this string.
- **`description`**: the only prose about the tool. The model uses it to decide *whether* this tool fits the question; with ten tools, this is how it picks one. Here the docstring and the description say different things, and only the description affects the model.
- **`parameters`**: a JSON Schema that tells the model which argument names and types to write. `"strict": true` makes the API constrain generation so the arguments always match this schema. It's the same constrained decoding you saw with `response_format` in Exhaustive 2.

##### How this relates to `Field(description=...)`
`Field` descriptions reach the model because the SDK converts your Pydantic class into a JSON Schema and puts it into the request body (you'll see it in Section 6). A plain function gets no such conversion; nothing reads its signature or docstring.

The tool equivalent of `Field(description=...)` is a `"description"` key inside each property. The course code has none, but you could write:
```python
"properties": {
    "latitude": {"type": "number", "description": "Latitude in decimal degrees, e.g. 59.91 for Oslo."},
    "longitude": {"type": "number", "description": "Longitude in decimal degrees, e.g. 10.75 for Oslo."},
},
```

Two related facts, so the rule doesn't get over-simplified:
- A docstring on a **Pydantic class** *is* sent, because Pydantic copies it into the schema as the top-level `"description"`. If you add `"""Final answer about the weather."""` under `class WeatherResponse(BaseModel):`, the schema the SDK sends starts with `{'description': 'Final answer about the weather.', 'properties': {...`.
- Some agent frameworks (e.g. the OpenAI Agents SDK's `@function_tool`) build the tool definition *from* a function's signature and docstring. The docstring reaches the model there only because the framework copies it into `"description"`. The plain `OpenAI()` client used here doesn't do that.

In [7]:
messages = [
    {"role": "system", "content": "You are a helpful weather assistant."},
    {"role": "user", "content": "What's the weather like in Oslo today?"},
]

completion = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
)

In [10]:
print(type(completion))
print(completion.model_dump_json(indent=2))

<class 'openai.types.chat.chat_completion.ChatCompletion'>
{
  "id": "chatcmpl-EPjMoBPyvkoZAPAP9czmew4GPDHLB",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": [
          {
            "id": "call_jVJ67G2TzgPRcCCApHW4ya7r",
            "function": {
              "arguments": "{\"latitude\":59.9139,\"longitude\":10.7522}",
              "name": "get_weather"
            },
            "type": "function"
          }
        ]
      }
    }
  ],
  "created": 1789801298,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 289,
    "prompt_tokens": 150,
    "total_tokens": 439,
    "complet

In [11]:
pprint(completion.model_dump(), sort_dicts=False)

{'id': 'chatcmpl-EPjMoBPyvkoZAPAP9czmew4GPDHLB',
 'choices': [{'finish_reason': 'tool_calls',
              'index': 0,
              'logprobs': None,
              'message': {'content': None,
                          'refusal': None,
                          'role': 'assistant',
                          'annotations': [],
                          'audio': None,
                          'function_call': None,
                          'tool_calls': [{'id': 'call_jVJ67G2TzgPRcCCApHW4ya7r',
                                          'function': {'arguments': '{"latitude":59.9139,"longitude":10.7522}',
                                                       'name': 'get_weather'},
                                          'type': 'function'}]}}],
 'created': 1789801298,
 'model': 'gpt-5-nano-2025-08-07',
 'object': 'chat.completion',
 'metadata': None,
 'moderation': None,
 'service_tier': 'default',
 'system_fingerprint': None,
 'usage': {'completion_tokens': 289,
           'prompt

#### 2. Reading the response: the model *asked* for a function, and nothing ran

**Key point:** the model's entire reply to call 1 is the request "run `get_weather` with `{"latitude":59.9139,"longitude":10.7522}`". There is no answer text (`content=None`), and `finish_reason='tool_calls'` says the model stopped because it wants a tool run. Everything else in the output is bookkeeping.

The two outputs above are **the same data in two forms**. Both come from Pydantic methods, which exist here because `ChatCompletion` is a Pydantic `BaseModel`:
- `completion.model_dump_json(indent=2)` returns a **string** of JSON (`null`, double quotes). `indent=2` puts one field per line.
- `completion.model_dump()` returns a Python **dict** (`None`, single quotes). `pprint` ("pretty print", from Python's standard library, imported at the top) prints it one key per line, and `sort_dicts=False` keeps the original key order instead of sorting the keys alphabetically.

A plain `print(completion)` would print the object in its own form, `ChatCompletion(id='chatcmpl-...', choices=[Choice(...)], ...)`, all on one line. The class names from that form are the ones in brackets in the tree below.

##### The tree (only the parts that matter; the rest are `None` or empty here)

<pre style="font-size: 0.85em; line-height: 1.4; white-space: pre; overflow-x: auto;">
completion  <span style="opacity: 0.55">(ChatCompletion)</span>
├─ id        'chatcmpl-EPjMoBPyvkoZAPAP9czmew4GPDHLB'
├─ created   1789801298                 <span style="opacity: 0.7">← Unix time</span>
├─ model     'gpt-5-nano-2025-08-07'    <span style="opacity: 0.7">← exact model version</span>
├─ choices   <span style="opacity: 0.55">(list, 1 item)</span>
│  └─ [0]  <span style="opacity: 0.55">(Choice)</span>
│     ├─ finish_reason  'tool_calls'    <span style="opacity: 0.7">← it wants a tool run</span>
│     └─ message  <span style="opacity: 0.55">(ChatCompletionMessage)</span>
│        ├─ role        'assistant'
│        ├─ content     None            <span style="opacity: 0.7">← no answer text</span>
│        └─ tool_calls  <span style="opacity: 0.55">(list, 1 item)</span>
│           └─ [0]  <span style="opacity: 0.55">(ChatCompletionMessageFunctionToolCall)</span>
│              ├─ id        'call_jVJ67G2TzgPRcCCApHW4ya7r'
│              ├─ type      'function'
│              └─ function  <span style="opacity: 0.55">(Function)</span>
│                 ├─ name       'get_weather'
│                 └─ arguments  '{"latitude":59.9139,"longitude":10.7522}'
└─ usage     <span style="opacity: 0.55">(CompletionUsage)</span>
   ├─ prompt_tokens      150            <span style="opacity: 0.7">← tokens read</span>
   ├─ completion_tokens  289            <span style="opacity: 0.7">← tokens written (256 reasoning)</span>
   └─ total_tokens       439
</pre>

The same path reaches the arguments string `'{"latitude":59.9139,"longitude":10.7522}'` in both forms:
```python
# object: dots
completion.choices[0].message.tool_calls[0].function.arguments

# dict: keys
d = completion.model_dump()
d["choices"][0]["message"]["tool_calls"][0]["function"]["arguments"]
```

Things worth noticing:
- **`finish_reason`** is `'tool_calls'` here. For a normal text answer it's `'stop'`.
- **`arguments` is in quotes:** it's a **string** that contains JSON, not a dict. Section 4 turns it into a dict with `json.loads`.
- **The tool call's `id`** (`'call_jVJ67G2TzgPRcCCApHW4ya7r'`) is how the result you send back gets paired with this request (Section 4).
- **Where the coordinates came from:** nothing in the request contained Oslo's latitude and longitude. The model supplied them from its training knowledge. That is the part of the job the model does: choosing the tool and filling in its arguments.
- **Reasoning tokens:** `gpt-5-nano` is a reasoning model, so it thinks privately before writing. Of the 289 tokens it wrote, 256 were that hidden reasoning (`usage.completion_tokens_details.reasoning_tokens`), and only the remaining 33 were the tool call itself. You pay for all 289.
- **The `None`/empty fields** belong to features not used here: `logprobs`, `audio`, `refusal` (filled if the model declines), `annotations` (e.g. web-search citations), and `function_call` (the older, deprecated form of `tool_calls`).

In [12]:
messages

[{'role': 'system', 'content': 'You are a helpful weather assistant.'},
 {'role': 'user', 'content': "What's the weather like in Oslo today?"}]

#### 3. Call 1 did not fetch the weather, so why the rest?

**Key point:** `tools` + `messages` + `create()` only get the model to *choose* a tool and *write its arguments*. They can't fetch anything, because the model runs on OpenAI's servers with no access to your function and can't execute Python. The weather is fetched by your code in the next cell. Then a second API call is needed so the model can read the result and write the answer.

Three pieces of evidence from the outputs above:
1. `content=None`: the model wrote no answer, only a tool request.
2. `messages` (the cell just above) still holds only the 2 messages you wrote. `create()` doesn't add anything to your list, and the API keeps no memory between calls. Whatever the model should see next time, *you* have to append.
3. Open-Meteo hasn't been contacted yet. The only code that sends a request to `api.open-meteo.com` is the body of `get_weather`. It runs on your machine, and nothing has called it so far.

So the roles are:
- **Completion 1** decides *which* function to run and *with which arguments*. It does not retrieve the information.
- **Your code** (the next cell) retrieves the information.
- **Completion 2** reads that information and writes the answer, here in the `WeatherResponse` shape.

#### 4. Running the tool ourselves: `call_function`, `**args`, and the loop

##### `call_function` is a dispatcher
**What it is:** a function that takes the tool name the model wrote and runs the matching Python function.

**Why it's needed:** the model only gives you the *string* `'get_weather'`, and you can't call a string. `call_function` maps the name to the real function. With one tool it's a single `if`. With several tools you add one branch per tool (or use a dict like `{"get_weather": get_weather}`).

##### `**args` unpacks a dict into keyword arguments
**What it does:** `**` inside a function call takes each `key: value` pair of a dict and passes it as `key=value`.

This checks that `json.loads` turns the model's arguments string into a dict:
```python
arguments = '{"latitude":59.9139,"longitude":10.7522}'
print(type(arguments))
args = json.loads(arguments)
print(args)
print(type(args))
```
```
<class 'str'>
{'latitude': 59.9139, 'longitude': 10.7522}
<class 'dict'>
```

This checks that `get_weather(**args)` is the same call as writing the keywords out by hand. It uses a stub `get_weather` that returns its inputs instead of calling the API:
```python
def get_weather(latitude, longitude):
    return f"called with latitude={latitude}, longitude={longitude}"

print(get_weather(**args))
print(get_weather(latitude=59.9139, longitude=10.7522))
```
```
called with latitude=59.9139, longitude=10.7522
called with latitude=59.9139, longitude=10.7522
```

This works only because the property names in the tool's `parameters` (`latitude`, `longitude`) exactly match the Python parameter names. This checks what happens when they don't:
```python
get_weather(**{"lat": 59.9139, "lon": 10.7522})
```
```
TypeError: get_weather() got an unexpected keyword argument 'lat'
```

##### The loop, line by line
`completion.choices[0].message.tool_calls` is a **list**, because the model can ask for several calls in one reply (e.g. "weather in Oslo and Paris?" gives two `get_weather` calls). The loop handles each one.

Before the loop, once:
- `messages.append(completion.choices[0].message)` adds the model's tool request (the whole assistant message from call 1) to the history. The API requires every `"role": "tool"` message to come after an assistant message whose `tool_calls` contains the matching id. Without it, call 2 would contain a result answering no request, and the API rejects that.

Inside the loop, once per tool call:
- `name = tool_call.function.name` gives `'get_weather'`, which your code uses to pick the function.
- `args = json.loads(tool_call.function.arguments)` turns the string into a dict (shown above) so Python can use it.
- `result = call_function(name, args)` is **where `get_weather` actually runs** and the real HTTP request goes to Open-Meteo. In the saved run (before the Section 7 fix) it returned `{'time': '2026-09-19T07:15', 'interval': 900, 'temperature_2m': 14.5, 'wind_speed_10m': 14.4}`. Now that dict comes back under `"current"`, next to `"units"` and `"timezone"`.
- `messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)})` hands the result back:
  - `"role": "tool"` marks the message as a tool result, not something the user said.
  - `"tool_call_id"` pairs the result with the request's id (`call_jVJ6...`). With several calls, this tells the model which result answers which request.
  - `json.dumps(result)` turns the dict into a string, because a message's `content` must be text.

The two conversions go in opposite directions: `json.loads` converts **string → dict** for your code, and `json.dumps` converts **dict → string** for the model.

##### Why the first append is above the loop (changed from the course code)
**Key point:** the model's request is one message per reply, not one per tool call. The course code appended it *inside* the loop. That's harmless with one tool call, but call 2 fails whenever the model asks for two or more tools in one reply.

I tested both versions against the real API with the question "What's the weather like in Oslo and in Paris today?". The model returned **2 tool calls in one reply**, and the two versions built these histories:
```
inside the loop (course)     before the loop (fixed)
system                       system
user                         user
assistant [Oslo, Paris]      assistant [Oslo, Paris]
tool      Oslo               tool      Oslo
assistant [Oslo, Paris]      tool      Paris
tool      Paris
```
- **Inside the loop:** call 2 was **rejected** with a 400 error. The first assistant message asks for two results, but only one tool message follows it before the next assistant message. The error named the Paris call's id:
  > An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_UEUm2GlZIM5muyhnnBa0Fjbs
- **Before the loop:** call 2 was **accepted**, and the model wrote its answer (`finish_reason` `'stop'`).

So it's not a deal breaker for this script's single-city question, but it's a real bug. Parallel tool calls are on by default, so any question that needs two lookups at once would crash call 2.

In [13]:
def call_function(name, args):
    if name == "get_weather":
        return get_weather(**args)


messages.append(completion.choices[0].message)  # the model's request: once, not once per tool call

for tool_call in completion.choices[0].message.tool_calls:
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)

    result = call_function(name, args)
    messages.append(
        {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)}
    )

In [14]:
messages

[{'role': 'system', 'content': 'You are a helpful weather assistant.'},
 {'role': 'user', 'content': "What's the weather like in Oslo today?"},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_jVJ67G2TzgPRcCCApHW4ya7r', function=Function(arguments='{"latitude":59.9139,"longitude":10.7522}', name='get_weather'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_jVJ67G2TzgPRcCCApHW4ya7r',
  'content': '{"time": "2026-09-19T07:15", "interval": 900, "temperature_2m": 14.5, "wind_speed_10m": 14.4}'}]

#### 5. The four messages the second call receives

**Key point:** `messages` is the model's entire memory. Call 2 reads all four entries, and entry 4 is the only place the weather data exists.

1. `system`: written by you.
2. `user`: written by you.
3. `assistant`: written by the model in call 1 and appended by you. It's a `ChatCompletionMessage` object, not a dict; the SDK accepts that and converts it to JSON when sending.
4. `tool`: written by you. It holds `get_weather`'s result as a string, and its `tool_call_id` matches the id in entry 3.

This is the `messages` part of call 2's request body, captured the same way as in Section 1:

<div style="font-size: 0.85em">

```json
[
  {"role": "system", "content": "You are a helpful weather assistant."},
  {"role": "user", "content": "What's the weather like in Oslo today?"},
  {
    "content": null,
    "role": "assistant",
    "tool_calls": [
      {
        "id": "call_jVJ67G2TzgPRcCCApHW4ya7r",
        "function": {
          "arguments": "{\"latitude\":59.9139,\"longitude\":10.7522}",
          "name": "get_weather"
        },
        "type": "function"
      }
    ]
  },
  {
    "role": "tool",
    "tool_call_id": "call_jVJ67G2TzgPRcCCApHW4ya7r",
    "content": "{\"time\": \"2026-09-19T07:15\", \"interval\": 900, \"temperature_2m\": 14.5, \"wind_speed_10m\": 14.4}"
  }
]
```

</div>

The `\"` are escaped quotes: the arguments and the tool result are strings that contain JSON, sitting inside the outer JSON.

The SDK sends only the fields that were set on the entry-3 object. Depending on what the API returned, empty fields like `"refusal": null` can also appear; they carry no information.

In [15]:
class WeatherResponse(BaseModel):
    temperature: float = Field(
        description="The current temperature in celsius for the given location."
    )
    response: str = Field(
        description="A natural language response to the user's question."
    )


#### 6. Call 2: the model writes the answer, and `Field()` descriptions get read

**Key point:** call 2 sends the four messages, the same `tools`, and the `WeatherResponse` schema. The model now has the weather data (entry 4), so it writes the final answer instead of asking for a tool.

This is the `response_format` part of call 2's request body. It's how your `Field(description=...)` text reaches the model:

<div style="font-size: 0.85em">

```json
"response_format": {
  "type": "json_schema",
  "json_schema": {
    "schema": {
      "properties": {
        "temperature": {
          "description": "The current temperature in celsius for the given location.",
          "title": "Temperature",
          "type": "number"
        },
        "response": {
          "description": "A natural language response to the user's question.",
          "title": "Response",
          "type": "string"
        }
      },
      "required": ["temperature", "response"],
      "title": "WeatherResponse",
      "type": "object",
      "additionalProperties": false
    },
    "name": "WeatherResponse",
    "strict": true
  }
}
```

</div>

**Why `tools=tools` is passed again:** the API stores nothing between calls, so each request has to describe everything again. If you want the model to still know `get_weather` exists (and be able to call it again), its definition has to be in this request too.

**What the code assumes:** call 2 could come back with *another* tool request instead of an answer. Then `finish_reason` would be `'tool_calls'` and `.parsed` would be `None`. This course code assumes one round trip is enough. Real agents repeat steps 2 and 3 of the flow in a loop until `finish_reason` is `'stop'`.

In [16]:
completion_2 = client.chat.completions.parse(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
    response_format=WeatherResponse,
)

In [17]:
final_response = completion_2.choices[0].message.parsed
print(final_response.temperature)
print(final_response.response)

14.5
The current temperature in Oslo is 14.5°C as of 07:15 local time, with winds around 14 m/s.


#### 7. Check the answer against what the model was given

**Key point:** the model only knows what's in `messages`. In the saved run, `get_weather` returned only `data["current"]`, which drops the units and the timezone from Open-Meteo's reply, and the final answer got both of them wrong. The code has since been fixed (see the end of this section).

What the model received (entry 4), compared with what a direct request to Open-Meteo shows the full reply also contains:
- `"wind_speed_10m": 14.4` arrived **with no unit**. The full reply has `"current_units": {..., "wind_speed_10m": "km/h"}`. The model guessed **m/s**, which is 3.6 times too large: 14.4 km/h is 4 m/s.
- `"time": "2026-09-19T07:15"` arrived **with no timezone**. The full reply has `"timezone": "GMT"`. The model called it "07:15 local time", but Oslo in September is on UTC+2 (summer time), so it was 09:15 there.
- The temperature is right because both the tool's `description` and the `Field` description say "celsius", and `temperature_2m` is in °C.

##### The fix (in the code since 2026-09-20)
There are two changes to `get_weather`, and both are needed:
1. **`&timezone=auto&wind_speed_unit=ms` in the URL** makes Open-Meteo send the time in the location's local time and the wind speed in m/s. On its own, this only changes the numbers. `data["current"]` still carries no labels, so the model would still be guessing, and would only be right by luck.
2. **Returning `data["current_units"]` and `data["timezone"]`** next to `data["current"]` puts the labels into the tool result, so the model reads them. With the new URL, a direct request for Oslo returns `"current_units": {..., "wind_speed_10m": "m/s"}` and `"timezone": "Europe/Oslo"`.

It's the same lesson as Section 1: if the model should know something, it has to be in the request.